# 🚀 M8 Optuna HPO on GPU (Pavagada Solar + Demand Forecasting)

### Instructions:
1. Make sure **Runtime → Change runtime type → GPU (T4)** is selected.
2. Click **Runtime → Run all** (or run each cell top to bottom).

In [ ]:
# Cell 1: Verify GPU & Reset Directory
import os, sys
os.chdir('/content')
import torch

print(f'PyTorch Version: {torch.__version__}')
if torch.cuda.is_available():
    print(f'✅ GPU is ready: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> select T4 GPU.')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content')

In [ ]:
# Cell 3: Locate and Extract colab_upload.zip
import os, sys, zipfile

os.chdir('/content')
project_dir = '/content/MAJOR_PROJECT'
os.makedirs(project_dir, exist_ok=True)

# Automatically find colab_upload.zip in Google Drive
candidates = [
    '/content/drive/MyDrive/MAJOR_PROJECT/colab_upload.zip',
    '/content/drive/MyDrive/colab_upload.zip'
]

zip_path = None
for c in candidates:
    if os.path.exists(c):
        zip_path = c
        break

if not zip_path:
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'colab_upload.zip' in files:
            zip_path = os.path.join(root, 'colab_upload.zip')
            break

if not zip_path:
    raise FileNotFoundError(
        '❌ colab_upload.zip could not be found anywhere in Google Drive! '
        'Please verify that colab_upload.zip is uploaded to your Google Drive.'
    )

print(f'✅ Found zip at: {zip_path}')
print('Extracting files...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(project_dir)

# If extracted with a subfolder wrapper, find the real root with 'src'
real_root = project_dir
for root, dirs, files in os.walk(project_dir):
    if 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'config.py')):
        real_root = root
        break

print(f'✅ Project extracted to: {real_root}')
print('Contents:', os.listdir(real_root))

In [ ]:
# Cell 4: Install Required Packages
import os
os.chdir('/content')
!pip install -q optuna lightgbm pyarrow pandas scikit-learn joblib
print('✅ Dependencies installed!')

In [ ]:
# Cell 5: Run Optuna HPO (GPU-Accelerated)
import sys, os, json, time, shutil
import pandas as pd
import torch

# 1. Locate project root containing 'src'
project_dir = '/content/MAJOR_PROJECT'
real_root = None
for root, dirs, files in os.walk(project_dir):
    if 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'config.py')):
        real_root = root
        break

if not real_root:
    raise FileNotFoundError('Could not locate the folder containing src/config.py!')

os.chdir(real_root)
if real_root not in sys.path:
    sys.path.insert(0, real_root)

print(f'Working directory: {real_root}')

# 2. Import project modules
from src.config import PROBLEMS, PROCESSED_DATA_DIR, RESULTS_MODELS_DIR, RESULTS_TABLES_DIR
from src.tuning import tune_dl_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Execution device: {device}')

# 3. Check for existing checkpoint (loads 10 already completed studies)
checkpoint_path = RESULTS_MODELS_DIR / 'optuna_best_params.json'
csv_path = RESULTS_TABLES_DIR / 'm8_hpo_results.csv'

if checkpoint_path.exists():
    with open(checkpoint_path) as f:
        optuna_results = json.load(f)
    print(f'✅ Checkpoint loaded: {len(optuna_results)}/20 studies already complete.')
else:
    optuna_results = {}
    print('No checkpoint found. Starting from study 1.')

# Google Drive sync folder
drive_results_dir = '/content/drive/MyDrive/MAJOR_PROJECT/results'
os.makedirs(f'{drive_results_dir}/models', exist_ok=True)
os.makedirs(f'{drive_results_dir}/tables', exist_ok=True)

# 4. Define target studies
DL_MODELS = ['bilstm', 'gru', 'cnn_lstm', 'dlinear', 'lstm']
problem_keys = list(PROBLEMS.keys())
total_studies = len(DL_MODELS) * len(problem_keys)
completed = len(optuna_results)

# 5. Run studies
for model_name in DL_MODELS:
    for pkey in problem_keys:
        study_key = f'{model_name}_{pkey}'

        if study_key in optuna_results:
            val = optuna_results[study_key].get('best_value', 0)
            print(f'[SKIP] {study_key:<16} already finished (Best CV Huber: {val:.5f})')
            continue

        print(f'\n==================================================')
        print(f'▶️ Tuning: {model_name} on {pkey} (Study {completed + 1}/{total_studies})')
        print(f'==================================================')
        t0 = time.time()

        best_params, best_value = tune_dl_model(
            model_name=model_name,
            problem_key=pkey,
            n_trials=35,
            device=device,
        )

        elapsed = time.time() - t0
        completed += 1

        print(f'  🎉 Completed {study_key} in {elapsed:.1f}s (~{elapsed/60:.1f} min)')
        print(f'  Best CV Huber: {best_value:.5f}')
        print(f'  Optimal Params: {best_params}')

        # Save locally
        optuna_results[study_key] = {
            'best_params': best_params,
            'best_value': float(best_value),
            'elapsed_seconds': float(elapsed),
        }
        with open(checkpoint_path, 'w') as f:
            json.dump(optuna_results, f, indent=2)

        # Save CSV
        rows = []
        for k, v in optuna_results.items():
            m, p = k.rsplit('_', 1)
            row = {'model': m, 'problem': p, 'cv_huber': v['best_value']}
            row.update(v['best_params'])
            rows.append(row)
        pd.DataFrame(rows).to_csv(csv_path, index=False)

        # Auto-sync to Google Drive
        shutil.copy2(str(checkpoint_path), f'{drive_results_dir}/models/optuna_best_params.json')
        shutil.copy2(str(csv_path), f'{drive_results_dir}/tables/m8_hpo_results.csv')
        print(f'  💾 Synced checkpoint directly to Google Drive!')

print('\n' + '='*50)
print('🎉 ALL 20 STUDIES COMPLETED SUCCESSFULLY!')
print(f'Final results saved in Google Drive: {drive_results_dir}')
print('='*50)

## ✅ What to do now:
1. Go to your Google Drive folder: `MAJOR_PROJECT/results/`
2. Download `models/optuna_best_params.json` and `tables/m8_hpo_results.csv`
3. Place them into your project on your laptop under `results/models/` and `results/tables/`
4. Return to the chat and say **"continue"**!